In [49]:
%pwd
os.chdir("/Users/aashuanand/AI-Powered-Precision-Agriculture-Monitoring-System")

In [23]:
%pwd

'/Users/aashuanand/AI-Powered-Precision-Agriculture-Monitoring-System'

In [47]:
import os 
os.chdir("../")

In [50]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/crop_health_assesment_dataset.csv")
df['date'] = pd.to_datetime(df['date'])

# pick one location
sample = df[
    (df['lat'] == df['lat'].iloc[0]) &
    (df['lon'] == df['lon'].iloc[0])
].copy()

sample = sample.sort_values("date")

ndvi_series = sample['NDVI'].values

In [51]:
# Group by lat/lon → get time series per pixel/point
ts = df.groupby(['lat', 'lon'])

# Each location gets a vector of ~52-70 values
# Shape: (n_locations, n_timestamps)

In [54]:
grouped = df.groupby(['lat', 'lon'])
ts_store={}
for (lat, lon), group in grouped:
    # ── Sort by date ──────────────────────────────────────────────────────
    group = group.sort_values('date').reset_index(drop=True)

    # ── Skip locations with too few observations ───────────────────────────
    if len(group) < 10:
        continue

    ts_store[(lat, lon)] = {
        'dates': group['date'].tolist(),
        'NDVI' : group['NDVI'].tolist(),
        'NDWI' : group['NDWI'].tolist(),
        'EVI'  : group['EVI'].tolist(),
        'SAVI' : group['SAVI'].tolist(),
    }

ts

In [16]:
def create_sequences(data, seq_length=5):
    X, y = [], []
    
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    
    return np.array(X), np.array(y)

seq_length = 5

X, y = create_sequences(ndvi_series, seq_length)

print(X.shape, y.shape)

(17, 5) (17,)


In [11]:
import tensorflow as tf

In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential([
    LSTM(32, input_shape=(seq_length, 1)),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 32)             │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,385 (17.13 KB)

 Trainable params: 4,385 (17.13 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
# reshape for LSTM
X = X.reshape((X.shape[0], X.shape[1], 1))

model.fit(X, y, epochs=20, verbose=1)

Epoch 1/20


2026-04-09 14:58:18.731372: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - loss: 0.1791
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 0.1677
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.1566
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.1459
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.1356
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - loss: 0.1256
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.1160
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.1068
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.0980
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.0896
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 0.0816
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0740
Epoch 13/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.0668
Epoch 14/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0600
Epoch 15/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.0536
Epoch 16/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/st

In [45]:
df = pd.read_csv("data/processed/crop_stress_labels.csv")

with pd.option_context('display.max_columns', None):
    print(df)

           lat        lon  ndvi_max  ndvi_min  ndvi_mean  ndvi_std  \
0    24.495982  84.526354  0.589734  0.130507   0.320384  0.135657   
1    24.495984  84.526302  0.585740  0.131672   0.310723  0.133416   
2    24.574162  83.632501  0.321851  0.133165   0.203595  0.052236   
3    24.664480  83.912392  0.608457  0.133070   0.353144  0.143559   
4    24.728383  85.507533  0.619159  0.043549   0.314444  0.195368   
..         ...        ...       ...       ...        ...       ...   
100  31.698126  75.958207  0.577196  0.265124   0.407497  0.086971   
101  31.761439  74.777559  0.672199  0.037219   0.367491  0.206117   
102  31.764143  75.309779  0.516866  0.039753   0.219524  0.155516   
103  31.788596  75.432600  0.612158  0.031169   0.310580  0.177681   
104  31.866280  75.743657  0.302173  0.150847   0.219357  0.044250   

     ndvi_range  ndvi_max_decline  ndvi_recovery  ndvi_below_thresh  \
0      0.459226         -0.210159       0.167879                 37   
1      0.454067  